Ask anyone what Las Vegas runs on and they'll say gambling. But the Nevada Gaming Control Board's own latest figure says gaming is only **26%** of Strip casino revenue. I wanted to know whether that's true, when it changed, and what took gambling's place.

> **The question.** Is Las Vegas still a gambling town?

| Source | What it has | Coverage |
|---|---|---|
| **Nevada Gaming Control Board, Gaming Abstract** | Strip casinos' income statements by department | Annual, FY1990 – FY2025 |
| **Nevada Gaming Control Board, monthly reports** | Gaming win by area and game | Monthly, Jan 2010 – Sep 2026 |
| **LVCVA Visitor Profile Study** | Whether visitors gambled, and what they budgeted and spent | Annual survey, 2003 – 2025 |
| **Bureau of Labor Statistics** | Consumer price index (CPI-U) | Monthly, 1989 on |

The Abstract is the board's yearly summary of casino financial statements. For Strip casinos with at least \$1 million in gaming revenue (32 to 54 casinos, depending on the year), it reports revenue from gaming, rooms, food, beverage and everything else. Fiscal years run July to June. **Money is in 2025 dollars** wherever I compare years.

In [1]:
#| collapse
import duckdb
import numpy as np
import pandas as pd
import altair as alt

con = duckdb.connect("../data/vegas.duckdb", read_only=True)
q = lambda sql: con.sql(sql).df()          # run SQL, get a DataFrame

GREEN, INK, MUTED, LIGHT = "#00934f", "#0c0c0c", "#66645e", "#b5b1a6"
q("SELECT min(fy) AS first_fy, max(fy) AS latest_fy, count(*) AS years FROM v_strip_departments").set_axis(["Gaming Abstract"])

,first_fy,latest_fy,years
Gaming Abstract,1990,2025,36


## Reading 36 years of scanned reports

Every Abstract before FY2018 is a scanned image of a printed page, so I used OCR (optical character recognition) to read them. OCR makes mistakes with numbers: it reads commas as digits and picks up specks as extra characters. One wrong digit in the billions would quietly ruin the analysis, so I only kept a reading if it passed checks the page itself allows:

1. The five departments have to add up exactly to the printed total.
2. Each department has to match its printed percentage, to a tenth of a point.
3. For profit, a department's income plus its expenses has to equal its revenue.

If a number failed, I left it blank instead of guessing.

In [2]:
# How each year's figures were read, and whether they passed the checks
q("""
    SELECT CASE WHEN method = 'text' THEN 'Text PDF (FY2018 on)' ELSE 'Scanned, read by OCR' END AS "Source",
           count(*) AS "Years", count(total) AS "Passed the checks"
    FROM raw_abstract_strip GROUP BY 1 ORDER BY 1 DESC
""").set_index("Source")

,Years,Passed the checks
Source,,
Text PDF (FY2018 on),8,8
"Scanned, read by OCR",28,28


Every year's revenue passed. A few profit figures didn't, so the profit chart further down has small gaps.

## An accounting change hides in the middle

Reading the reports in order turned up something I didn't expect. In FY2019, gaming's share of Strip revenue drops from 34% to 27% in a single year, and gaming revenue falls by \$1.8 billion. Nothing happened to gambling that year; the monthly reports show Strip win was flat.

What changed was the accounting. Starting in FY2019, casinos adopted a new revenue rule (ASC 606). Before, when a casino gave a player a free room or meal (a "comp"), the gambling revenue counted in full and the comp showed up as a casino expense. Under the new rule, the value of the comp comes out of gaming revenue instead. Same money, smaller gaming number.

The Abstract still prints the comps it subtracted, so I added them back. That puts every year on the same basis. To check the result, I compared it with the monthly reports, which count gaming win directly:

In [3]:
# Old-basis gaming revenue from the Abstract against the monthly reports' Strip win, same fiscal year
q("""
    SELECT d.fy AS "Fiscal year",
           round(d.gaming / 1e9, 2) AS "Abstract gaming ($B)",
           round(w.win / 1e9, 2)    AS "Monthly reports' win ($B)",
           round(d.gaming / w.win, 2) AS "Ratio"
    FROM v_strip_departments d
    JOIN (SELECT year(month + INTERVAL 6 MONTH) AS fy, sum(total_win) AS win, count(*) AS n
          FROM fct_area_month WHERE area_key = 'strip' GROUP BY ALL) w USING (fy)
    WHERE w.n = 12 AND d.fy IN (2011, 2015, 2018, 2019, 2022, 2025)
    ORDER BY 1
""").set_index("Fiscal year")

,Abstract gaming ($B),Monthly reports' win ($B),Ratio
Fiscal year,,,
2011,5.50,6.01,0.91
2015,5.84,6.33,0.92
2018,6.29,6.60,0.95
2019,6.45,6.54,0.99
2022,7.86,8.14,0.97
2025,8.44,8.78,0.96


The Abstract's gaming revenue comes to 91–99% of the monthly win in every year. It only covers the larger casinos, so it should come in a little lower, and it does.

In [4]:
#| hide
#| caption: Gaming's share of Strip casino revenue as the board prints it (grey) and on one consistent basis (green). The two match until FY2019, when a new accounting rule took comps out of gaming revenue.
d = q("SELECT fy, gaming_share_reported AS reported, gaming_share AS consistent FROM v_strip_departments ORDER BY fy")
long = d.melt("fy", ["reported", "consistent"], "basis", "share")
names = {"consistent": "Consistent basis (comps added back)", "reported": "As printed"}
long["basis"] = long.basis.map(names)
alt.Chart(long).mark_line(strokeWidth=2.4, point=alt.OverlayMarkDef(size=28, filled=True)).encode(
    x=alt.X("fy:O", title="Fiscal year", axis=alt.Axis(labelAngle=0, values=list(range(1990, 2026, 5)))),
    y=alt.Y("share:Q", title="Gaming's share of revenue", axis=alt.Axis(format="%"), scale=alt.Scale(domain=[0.2, 0.6])),
    color=alt.Color("basis:N", title=None, scale=alt.Scale(domain=list(names.values()), range=[GREEN, LIGHT])),
    tooltip=[alt.Tooltip("fy:O", title="Fiscal year"), "basis:N", alt.Tooltip("share:Q", format=".1%", title="Share")],
).properties(title="The printed share drops in 2019; the consistent one doesn't", height=300)

alt.Chart(...)

So the board's 26% for FY2025 is accurate, but it isn't comparable with the older figures. On the old basis, gaming was **about 35% of Strip revenue in FY2025**, the same as in FY2015.

## Gambling's share fell for 25 years, then stopped

In [5]:
share = q("""
    SELECT fy, round(gaming_share, 3) AS "Gaming share", round(gaming * to_2025_dollars / 1e9, 1) AS "Gaming ($B, 2025)",
           round((total - gaming) * to_2025_dollars / 1e9, 1) AS "Everything else ($B, 2025)"
    FROM v_strip_departments WHERE fy IN (1990, 1995, 2000, 2005, 2010, 2015, 2019, 2023, 2025) ORDER BY fy
""").set_index("fy")
share

,Gaming share,"Gaming ($B, 2025)","Everything else ($B, 2025)"
fy,,,
1990,0.578,5.8,4.2
1995,0.538,7.5,6.5
2000,0.459,8.9,10.5
2005,0.409,8.9,12.8
2010,0.390,7.7,12.0
2015,0.349,7.9,14.8
2019,0.345,8.2,15.6
2023,0.350,8.9,16.5
2025,0.347,8.6,16.1


In [6]:
#| hide
#| caption: Strip casino revenue by department, fiscal years 1990 to 2025, in 2025 dollars, on the consistent basis. FY2020–21 are the COVID closure and reopening.
order = ["gaming", "rooms", "food", "beverage", "other"]
labels = {"gaming": "Gaming", "rooms": "Rooms", "food": "Food", "beverage": "Beverage", "other": "Other (shows, retail, etc.)"}
dr = q("SELECT fy, dept, real_revenue / 1e9 AS revenue_b FROM v_strip_departments_real")
dr["dept"] = dr.dept.map(labels)
dr["order"] = dr.dept.map({labels[k]: i for i, k in enumerate(order)})
alt.Chart(dr).mark_area().encode(
    x=alt.X("fy:O", title="Fiscal year", axis=alt.Axis(labelAngle=0, values=list(range(1990, 2026, 5)))),
    y=alt.Y("revenue_b:Q", title="Revenue ($B, 2025 dollars)", stack="zero"),
    color=alt.Color("dept:N", title=None, sort=[labels[k] for k in order],
                    scale=alt.Scale(domain=[labels[k] for k in order], range=[GREEN, INK, "#1d5bff", "#eda100", LIGHT]),
                    legend=alt.Legend(columns=2)),
    order=alt.Order("order:Q"),
    tooltip=[alt.Tooltip("fy:O", title="Fiscal year"), alt.Tooltip("dept:N", title="Department"),
             alt.Tooltip("revenue_b:Q", format=".2f", title="Revenue ($B)")],
).properties(title="Gaming grew a little; everything else grew a lot", height=320)

alt.Chart(...)

In FY1990, gaming was **58%** of Strip casino revenue. By FY2015 it was **35%**, and it has stayed between 34% and 38% in every normal year since. (FY2021 jumps to 48% because casinos reopened before hotels, shows and restaurants were back to normal.)

The share fell because everything else grew, not because gambling shrank. In 2025 dollars, gaming revenue went from about \$5.8 billion in FY1990 to \$8.6 billion in FY2025. Everything else went from \$4.2 billion to \$16.1 billion, nearly four times as much. Rooms led the way, from \$1.7 billion to \$7.3 billion.

Gaming has been close to flat for a while, too. It peaked at \$10.2 billion in FY2007, fell in the recession, and has stayed between about \$7.7 and \$9.1 billion in every normal year since 2010.

## The rooms now earn the profit

Revenue isn't profit, so I also pulled each department's income: its revenue minus its own costs (staff, comps, gaming taxes), before company-wide overhead like interest and depreciation.

In [7]:
#| hide
#| caption: Departmental income (revenue minus the department's own costs, before overhead) for the casino and the rooms, in 2025 dollars. Gaps are years where the scanned figure couldn't be read and checked.
p = q("SELECT fy, casino_dept_income * to_2025_dollars / 1e9 AS casino, rooms_dept_income * to_2025_dollars / 1e9 AS rooms FROM v_strip_departments ORDER BY fy")
pl = p.melt("fy", ["casino", "rooms"], "dept", "income")
pl["dept"] = pl.dept.map({"casino": "Casino", "rooms": "Rooms"})
alt.Chart(pl).mark_line(strokeWidth=2.4, point=alt.OverlayMarkDef(size=28, filled=True)).encode(
    x=alt.X("fy:O", title="Fiscal year", axis=alt.Axis(labelAngle=0, values=list(range(1990, 2026, 5)))),
    y=alt.Y("income:Q", title="Departmental income ($B, 2025 dollars)"),
    color=alt.Color("dept:N", title=None, scale=alt.Scale(domain=["Casino", "Rooms"], range=[GREEN, INK])),
    tooltip=[alt.Tooltip("fy:O", title="Fiscal year"), alt.Tooltip("dept:N", title="Department"),
             alt.Tooltip("income:Q", format=".2f", title="Income ($B)")],
).properties(title="Rooms now earn more than the casino floor", height=280)

alt.Chart(...)

**Casino departmental income has barely moved in 35 years**: about \$2.7 billion in FY1990 and \$2.8 billion in FY2025, in 2025 dollars. Rooms went from \$0.9 billion to \$4.9 billion. Rooms first out-earned the casino in FY2001, and they have every year since FY2007 except the COVID year.

Part of the reason is comps. Casinos give back a growing share of what they win as free rooms, meals and drinks:

In [8]:
comps = q("""
    SELECT fy, round(comps_share_of_win, 3) AS "Comps as share of gaming win"
    FROM v_strip_departments WHERE fy IN (1990, 2000, 2010, 2018, 2019, 2025) ORDER BY fy
""").set_index("fy")
comps

,Comps as share of gaming win
fy,
1990,0.133
2000,0.170
2010,0.269
2018,0.259
2019,0.308
2025,0.348


Comps were about 13% of gaming win in the early 1990s and about 26% by the 2010s. (The new rule values comps differently, so the figures from FY2019 on, around 31–35%, aren't directly comparable.) Comps count as a casino cost, and the comped rooms show up as room revenue. In FY2018, for example, \$0.86 billion of the \$5.1 billion in room revenue was comped rooms. So some of the rooms' profit is really the casino paying for the room.

## What visitors say

The Abstract shows where the money goes. The LVCVA's annual visitor survey shows what people say they do, so it's a useful second angle.

In [9]:
#| hide
#| caption: Share of visitors who gambled during their trip (LVCVA Visitor Profile Study). No survey in 2020; an online component was added from 2022, so later years aren't strictly comparable with earlier ones.
v = q("SELECT year, gambled_pct FROM v_visitor_profile ORDER BY year")
line = alt.Chart(v).mark_line(color=GREEN, strokeWidth=2.4, point=alt.OverlayMarkDef(size=36, filled=True, color=GREEN)).encode(
    x=alt.X("year:Q", title=None, axis=alt.Axis(format="d", values=list(range(2003, 2026, 2))), scale=alt.Scale(domain=[2003, 2025])),
    y=alt.Y("gambled_pct:Q", title="Visitors who gambled", axis=alt.Axis(format="%"), scale=alt.Scale(domain=[0.6, 0.95])),
    tooltip=[alt.Tooltip("year:Q", format="d", title="Year"), alt.Tooltip("gambled_pct:Q", format=".0%", title="Gambled")])
line.properties(title="Fewer visitors gambled through the 2010s, then more again", height=260)

alt.Chart(...)

The share of visitors who gambled fell from **88% in 2003 to 69% in 2016**, then climbed back to about **80%**. So visitors haven't given up gambling. From 2022, the survey added online responses to its in-person interviews, so the latest years aren't strictly comparable with earlier ones.

In [10]:
#| hide
#| caption: Average trip gambling budget (among visitors who gambled) and average trip spending on food and drink (all visitors), in 2025 dollars. LVCVA Visitor Profile Study; no survey in 2020.
s = q("SELECT year, real_gambling_budget AS gambling, real_food_drink AS food FROM v_visitor_profile ORDER BY year")
sl = s.melt("year", ["gambling", "food"], "kind", "dollars")
sl["kind"] = sl.kind.map({"gambling": "Gambling budget (gamblers)", "food": "Food and drink (all visitors)"})
alt.Chart(sl).mark_line(strokeWidth=2.4, point=alt.OverlayMarkDef(size=30, filled=True)).encode(
    x=alt.X("year:Q", title=None, axis=alt.Axis(format="d", values=list(range(2003, 2026, 2))), scale=alt.Scale(domain=[2003, 2025])),
    y=alt.Y("dollars:Q", title="Per trip (2025 dollars)"),
    color=alt.Color("kind:N", title=None, scale=alt.Scale(domain=["Gambling budget (gamblers)", "Food and drink (all visitors)"], range=[GREEN, INK])),
    tooltip=[alt.Tooltip("year:Q", format="d", title="Year"), alt.Tooltip("kind:N", title=""), alt.Tooltip("dollars:Q", format="$,.0f", title="Per trip")],
).properties(title="Gambling budgets shrank in real terms; food and drink spending grew", height=280)

alt.Chart(...)

Spending tells a clearer story. Among visitors who gambled, the average trip gambling budget was about \$1,040 in 2006, in 2025 dollars. It fell to about \$640 by 2011 and has recovered to about \$850, still about 18% below 2006. Spending on food and drink went the other way, from about \$365 a trip in 2003 to about \$630 in 2024. In 2006, a gambler's budget was 2.5 times what the average visitor spent on food and drink. Now it's about 1.5 times.

Both lines are survey answers, not receipts, and the gambling budget only counts visitors who gambled. So I'd read the gap for its direction, not its precise size.

## Why monthly gaming headlines are noisy

Gaming is fairly flat from year to year, but it's anything but steady from month to month, and most of that comes from one game. Baccarat is played for very high stakes by a small number of players. The house's long-run edge is small, so in any one month the result depends heavily on whether a few big players win or lose. The share of chips bought that the house ends up keeping is called the **hold**.

In [11]:
# How much does each line move from one month to the next? (median absolute change, 2022 on)
swing = q('''
    WITH g AS (
        SELECT game_group, month, win / LAG(win) OVER (PARTITION BY game_group ORDER BY month) - 1 AS change
        FROM v_game_mix
        WHERE area_key = 'strip' AND month >= '2022-01-01'
          AND game_group IN ('Baccarat', 'Blackjack', 'Multi-denomination slots')
    )
    SELECT game_group AS "Game group",
           median(abs(change))              AS "Typical monthly change",
           quantile_cont(abs(change), 0.9)  AS "1 month in 10 moves at least"
    FROM g GROUP BY ALL ORDER BY 2 DESC
''').set_index("Game group")
swing.round(2)

,Typical monthly change,1 month in 10 moves at least
Game group,,
Baccarat,0.29,0.97
Blackjack,0.17,0.44
Multi-denomination slots,0.07,0.16


In [12]:
#| label: Share of month-to-month variation in Strip win from baccarat, 2022 on
# How much of the Strip's month-to-month change comes from baccarat? (2022 on)
mv = q('''
    SELECT a.month, a.total_win, g.win AS baccarat
    FROM v_area_monthly a
    LEFT JOIN v_game_mix g ON g.month = a.month AND g.area_key = 'strip' AND g.game_group = 'Baccarat'
    WHERE a.area_key = 'strip' AND a.month >= '2022-01-01'
    ORDER BY a.month
''')
d = mv[["total_win", "baccarat"]].diff().dropna()   # months without game detail drop out
round(d.cov().loc["baccarat", "total_win"] / d["total_win"].var(), 2)

np.float64(0.52)

In [13]:
#| hide
#| caption: Monthly baccarat hold on the Strip (dots) against its trailing two-year average (line), 2010 to 2026. Hover for the month's baccarat win.
bac = q("SELECT month, hold, hold_trailing_24m, baccarat_win / 1e6 AS win_m FROM v_baccarat ORDER BY month")
pts = alt.Chart(bac).mark_circle(size=36, color=GREEN, opacity=0.85).encode(
    x=alt.X("month:T", title=None),
    y=alt.Y("hold:Q", title="Hold (share of chips kept)", axis=alt.Axis(format="%")),
    tooltip=[alt.Tooltip("month:T", format="%b %Y", title="Month"), alt.Tooltip("hold:Q", format=".1%", title="Hold"),
             alt.Tooltip("win_m:Q", format=",.0f", title="Baccarat win ($M)")])
line = alt.Chart(bac.dropna(subset=["hold_trailing_24m"])).mark_line(color=INK, strokeWidth=2).encode(x="month:T", y="hold_trailing_24m:Q")
(line + pts).properties(title="Baccarat hold has always swung, and the average has crept up", height=280)

alt.LayerChart(...)

Since 2022, baccarat win typically moves about **29%** from one month to the next, against **7%** for multi-denomination slots. Baccarat is about a sixth of Strip gaming win, but it accounts for **about half of the month-to-month variation** in the total. So when a monthly headline says "Strip gaming revenue fell," the first question to ask is **what did baccarat do?** The yearly Abstract figures above smooth most of this out.

## What it adds up to

1. **Las Vegas is still a gambling town, but less than it was.** Gaming fell from 58% of Strip casino revenue in FY1990 to about 35% by FY2015, and it has held there for a decade.
2. **The "quarter of revenue" figure is mostly an accounting change.** The FY2019 rule moved comps out of gaming revenue. On a consistent basis, the share hasn't fallen since 2015.
3. **The shift came from growth elsewhere, not a collapse in gambling.** Real gaming revenue is up about 50% since FY1990; everything else nearly quadrupled.
4. **The rooms now earn the profit.** Casino departmental income is about where it was in 1990 in real terms. Rooms earn about \$4.9 billion, roughly \$2 billion more than the casino.
5. **Visitors still gamble, but they budget less for it.** About 80% gambled in 2025, but real gambling budgets are below their mid-2000s level, while food and drink spending is up sharply.
6. **Read monthly gaming headlines through baccarat.** It's about a sixth of gaming win but half the month-to-month swings.

## What this does not show

- The Abstract covers Strip casinos with at least \$1 million in gaming revenue. That's nearly all of them (its gaming totals come to 91–99% of the monthly win), but the group changes a little from year to year as casinos open and close.
- Adding comps back keeps the shares comparable, but the new rule also values comps differently, so dollar amounts from FY2019 on aren't a perfect match for the older ones.
- Departmental income comes before overhead. Interest, depreciation and corporate costs aren't included, so it isn't a casino's bottom line.
- Some of the rooms' revenue and profit is comped rooms paid for by the casino, so the two departments aren't fully independent.
- The visitor survey is self-reported, the gambling budget only covers visitors who gambled, and an online component was added in 2022. No survey ran in 2020.
- Inflation uses the national CPI-U. Las Vegas prices may have moved differently.
- A few scanned profit figures couldn't be read and checked, so they're left blank.

The pipeline, OCR checks, SQL model and data tests are in the [project folder on GitHub](https://github.com/mckeeander/mckeeander.github.io/tree/main/projects/vegas-gaming). The month-by-month dashboard at the top of this page tracks gaming win as new reports come out.

*Sources: Nevada Gaming Control Board, Gaming Abstract (FY1990–FY2025) and Monthly Gaming Revenue Reports; Las Vegas Convention and Visitors Authority, Las Vegas Visitor Profile Study (2003–2025, via the Nevada State Library archive and lvcva.com); U.S. Bureau of Labor Statistics, CPI-U.*